# InfraMind AI - Enterprise Compliance & Modernization Engine

## Install Dependencies

In [ ]:
!pip install python-hcl2 networkx chromadb sentence-transformers transformers accelerate torch pandas matplotlib pypdf


## Imports

In [ ]:
import os
import re
import json
import torch
import hcl2
import chromadb
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM


## Configuration

In [ ]:
TERRAFORM_FOLDER='./terraform'
POLICY_DOCUMENT='./security_standards.pdf'
CHROMA_DB_PATH='./vectordb'
MODEL_NAME='REPLACE_WITH_MODEL_NAME'


## Parse Terraform

In [ ]:
terraform_data=[]

for tf_file in Path(TERRAFORM_FOLDER).rglob('*.tf'):
    try:
        with open(tf_file,'r') as f:
            parsed=hcl2.load(f)

        terraform_data.append({
            'file':str(tf_file),
            'content':parsed
        })
    except Exception as e:
        print(e)

print(f'Loaded {len(terraform_data)} files')


## Build Knowledge Graph

In [ ]:
graph=nx.DiGraph()

for tf in terraform_data:

    resources=tf['content'].get('resource',[])

    for resource in resources:
        for resource_type, values in resource.items():
            for resource_name, config in values.items():

                node=f'{resource_type}.{resource_name}'
                graph.add_node(node)

                config_text=json.dumps(config)

                deps=re.findall(
                    r'aws_[A-Za-z0-9_]+\.[A-Za-z0-9_]+',
                    config_text
                )

                for dep in deps:
                    graph.add_edge(node,dep)

print('Nodes:',graph.number_of_nodes())
print('Edges:',graph.number_of_edges())


## Visualize Architecture

In [ ]:
plt.figure(figsize=(12,8))
pos=nx.spring_layout(graph)
nx.draw(graph,pos,with_labels=True,node_size=1500,font_size=8)
plt.show()


## ChromaDB Setup

In [ ]:
client=chromadb.PersistentClient(path=CHROMA_DB_PATH)

terraform_collection=client.get_or_create_collection('terraform_docs')
policy_collection=client.get_or_create_collection('company_policies')

embedding_model=SentenceTransformer('all-MiniLM-L6-v2')


## Index Terraform

In [ ]:
for idx,tf in enumerate(terraform_data):

    doc=json.dumps(tf['content'],default=str)

    embedding=embedding_model.encode(doc)

    terraform_collection.add(
        ids=[str(idx)],
        embeddings=[embedding.tolist()],
        documents=[doc]
    )


## Load Company Security Standards

In [ ]:
reader=PdfReader(POLICY_DOCUMENT)

policy_text=''

for page in reader.pages:
    policy_text += page.extract_text() + '\n'

print('Policy Loaded')


## Index Policy Document

In [ ]:
def chunk_text(text,size=1000):
    return [text[i:i+size] for i in range(0,len(text),size)]

chunks=chunk_text(policy_text)

for idx,chunk in enumerate(chunks):

    embedding=embedding_model.encode(chunk)

    policy_collection.add(
        ids=[f'policy_{idx}'],
        embeddings=[embedding.tolist()],
        documents=[chunk]
    )


## RAG Retrieval

In [ ]:
def retrieve_terraform_context(query,top_k=5):

    embedding=embedding_model.encode(query)

    result=terraform_collection.query(
        query_embeddings=[embedding.tolist()],
        n_results=top_k
    )

    return '\n'.join(result['documents'][0])

def retrieve_policy_context(query,top_k=5):

    embedding=embedding_model.encode(query)

    result=policy_collection.query(
        query_embeddings=[embedding.tolist()],
        n_results=top_k
    )

    return '\n'.join(result['documents'][0])


## Load LLM

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

model=AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto'
)


## LLM Helper

In [ ]:
def ask_llm(prompt):

    inputs=tokenizer(
        prompt,
        return_tensors='pt'
    ).to(model.device)

    outputs=model.generate(
        **inputs,
        max_new_tokens=2048
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


## Security Agent

In [ ]:
def run_security_agent():

    context=retrieve_terraform_context(
        'security vulnerabilities'
    )

    prompt=f'''
Analyze Terraform.

Find:
1. Security Issues
2. Severity
3. Risk
4. Terraform Fix

Terraform:
{context}
'''

    return ask_llm(prompt)


## CIS Agent

In [ ]:
def run_cis_agent():

    context=retrieve_terraform_context(
        'CIS AWS Foundations'
    )

    prompt=f'''
Review Terraform against CIS AWS Foundations Benchmark.

Provide:
- Control ID
- Finding
- Severity
- Terraform Fix

{context}
'''

    return ask_llm(prompt)


## NIST Agent

In [ ]:
def run_nist_agent():

    context=retrieve_terraform_context(
        'NIST review'
    )

    prompt=f'''
Review Terraform against NIST 800-53.

Provide:
- Control ID
- Finding
- Risk
- Terraform Fix

{context}
'''

    return ask_llm(prompt)


## Internal Policy Agent

In [ ]:
def run_internal_policy_agent():

    terraform_context=retrieve_terraform_context(
        'infrastructure'
    )

    policy_context=retrieve_policy_context(
        'security standards'
    )

    prompt=f'''
Evaluate Terraform against company security standards.

Policy:
{policy_context}

Terraform:
{terraform_context}

Provide:
- Violated Requirement
- Severity
- Risk
- Terraform Remediation
'''

    return ask_llm(prompt)


## Well Architected Agent

In [ ]:
def run_waf_agent():

    context=retrieve_terraform_context(
        'well architected'
    )

    prompt=f'''
Evaluate Terraform using AWS Well Architected Framework:

1. Operational Excellence
2. Security
3. Reliability
4. Performance Efficiency
5. Cost Optimization
6. Sustainability

Terraform:
{context}
'''

    return ask_llm(prompt)


## Migration Agent

In [ ]:
def run_migration_agent():

    context=retrieve_terraform_context(
        'aws to gcp migration'
    )

    prompt=f'''
Convert AWS Terraform to GCP.

Provide:
- Resource Mapping
- Complexity
- Equivalent GCP Terraform

{context}
'''
    return ask_llm(prompt)


## Cost Optimization Agent

In [ ]:
def run_cost_agent():

    context=retrieve_terraform_context(
        'cost optimization'
    )

    prompt=f'''
Analyze infrastructure cost.

Provide:
- Savings Opportunities
- Rightsizing
- Recommendations

{context}
'''
    return ask_llm(prompt)


## Modernization Agent

In [ ]:
def run_modernization_agent():

    context=retrieve_terraform_context(
        'modernization'
    )

    prompt=f'''
Suggest:
- Containerization
- Serverless
- Managed Services
- Architecture Improvements

{context}
'''
    return ask_llm(prompt)


## Compliance Manager Agent

In [ ]:
def run_compliance_manager(cis,nist,internal):

    prompt=f'''
Combine findings.

Inputs:
1. CIS
2. NIST
3. Internal Policy

Remove duplicates.
Prioritize risks.
Generate executive summary.

CIS:
{cis}

NIST:
{nist}

Internal:
{internal}
'''
    return ask_llm(prompt)


## Execute Agents

In [ ]:
security_report=run_security_agent()
cis_report=run_cis_agent()
nist_report=run_nist_agent()
internal_policy_report=run_internal_policy_agent()
well_architected_report=run_waf_agent()
migration_report=run_migration_agent()
cost_report=run_cost_agent()
modernization_report=run_modernization_agent()

compliance_summary=run_compliance_manager(
    cis_report,
    nist_report,
    internal_policy_report
)


## Final Report

In [ ]:
final_report={
    'security':security_report,
    'cis':cis_report,
    'nist':nist_report,
    'internal_policy':internal_policy_report,
    'well_architected':well_architected_report,
    'compliance_summary':compliance_summary,
    'migration':migration_report,
    'cost':cost_report,
    'modernization':modernization_report
}

print(json.dumps(final_report,indent=2))
